# LeetCode 487 — Max Consecutive Ones II bằng Python

Link đề: https://leetcode.com/problems/max-consecutive-ones-ii/ · Độ khó: **Medium** · Chủ đề: Array, Sliding Window, Dynamic Programming, Xử lý luồng dữ liệu (stream)

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode. Đề trong notebook được viết lại theo hiểu biết về bài, nên hãy đối chiếu lại đề gốc nếu bạn có tài khoản Premium. Ví dụ phụ do notebook tự dựng được ghi rõ là "tự dựng".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng nhị phân `nums` (chỉ `0` và `1`); được **lật tối đa một số `0` thành `1`**; tìm độ dài lớn nhất của đoạn gồm toàn số `1` liên tiếp.
2. Dùng được các công cụ Python: `enumerate`, hai chỉ số `l`/`r` làm **cửa sổ trượt**, phép XOR `x ^ 1` để đảo bit, danh sách đếm tiền tố, và **generator/iterator** để mô phỏng luồng dữ liệu.
3. Nắm kỹ thuật **cửa sổ trượt (sliding window) với ràng buộc "tối đa một số 0"**, và biến thể **cửa sổ không co** chỉ cần một biến đếm.
4. Cài đặt 5 cách khác nhau về ý tưởng: **thử lật từng số 0 → mảng độ dài chuỗi `1` bên trái/phải → cửa sổ trượt co khi cần → cửa sổ không bao giờ co → bộ đếm hai trạng thái cho luồng dữ liệu (follow-up)**, và so sánh độ phức tạp.
5. Tránh các lỗi kinh điển: dùng `max_run + 1` (quên giới hạn `n`), nhầm điều kiện co cửa sổ (`> 1` thay vì `>= 1`), truy cập `left[-1]` (Python **vòng** chỉ số âm!), và quên cộng chính số `0` bị lật.

## 2. Đề bài & ví dụ

Cho mảng nhị phân `nums`, trả về **số lượng số `1` liên tiếp lớn nhất** trong mảng, nếu bạn có thể **lật nhiều nhất một số `0`** (đổi `0` thành `1`).

**Ví dụ 1:** `nums = [1, 0, 1, 1, 0]` → `4`.

- Lật số `0` đầu tiên: `[1, 1, 1, 1, 0]` có 4 số `1` liên tiếp.
- Lật số `0` thứ hai: `[1, 0, 1, 1, 1]` có 3 số `1` liên tiếp.
- Lớn nhất là `4`.

**Ví dụ 2:** `nums = [1, 0, 1, 1, 0, 1]` → `4`.

- Lật số `0` đầu tiên: `[1, 1, 1, 1, 0, 1]` có 4 số `1` liên tiếp.
- Lật số `0` thứ hai: `[1, 0, 1, 1, 1, 1]` có 4 số `1` liên tiếp.
- Lớn nhất là `4`.

**Ràng buộc:** `1 <= nums.length <= 10^5`; `nums[i]` là `0` hoặc `1`.

**Follow up (câu hỏi mở rộng của đề):** nếu các số **đến từng cái một như một luồng vô hạn** (ta không thể lưu cả mảng vì quá lớn) thì sao? Có giải được hiệu quả không? (Ta trả lời ở Cách 5.)

In [1]:
vi_du = [([1, 0, 1, 1, 0], 4), ([1, 0, 1, 1, 0, 1], 4)]
for nums, kq in vi_du:
    print(f"nums = {nums} -> kỳ vọng {kq}")

nums = [1, 0, 1, 1, 0] -> kỳ vọng 4
nums = [1, 0, 1, 1, 0, 1] -> kỳ vọng 4


## 3. Kiến thức nền

### 3.1. `enumerate` và tìm vị trí các số `0`

`enumerate(a)` cho ta cặp `(chỉ số, giá trị)`. Ta dùng để liệt kê **chỉ số các số `0`** (chỉ những chỗ đó mới đáng lật).

In [2]:
nums = [1, 0, 1, 1, 0]
vi_tri_0 = [i for i, x in enumerate(nums) if x == 0]
print(vi_tri_0)       # [1, 4]

[1, 4]


### 3.2. Đoạn con liên tiếp và "cửa sổ" `[l, r]`

Một **đoạn con liên tiếp** của mảng là `nums[l..r]`. Ta gọi nó là **cửa sổ** với hai đầu `l` (trái) và `r` (phải); độ dài là `r - l + 1`. Với bài này, ta muốn cửa sổ **dài nhất** có **tối đa một số `0`**: vì số `0` duy nhất đó sẽ bị lật thành `1`, cả cửa sổ trở thành toàn `1`.

In [3]:
nums = [1, 0, 1, 1, 0, 1]
l, r = 0, 3
cua_so = nums[l:r + 1]
print(cua_so, "độ dài", r - l + 1, "số 0:", cua_so.count(0))   # [1,0,1,1] độ dài 4, 1 số 0

[1, 0, 1, 1] độ dài 4 số 0: 1


### 3.3. Đảo bit bằng `x ^ 1` và đếm số `0` bằng cộng dồn

Phép XOR `^` với `1` **đảo bit**: `0 ^ 1 = 1`, `1 ^ 1 = 0`. Vậy `x ^ 1` bằng `1` đúng khi `x == 0`, nên `cnt += x ^ 1` đếm số `0` đã đi qua. (Cách thông thường: `if x == 0: cnt += 1`. Cả hai tương đương.)

In [4]:
nums = [1, 0, 1, 1, 0]
cnt = 0
for x in nums:
    cnt += x ^ 1
    print(f"x={x}, x^1={x ^ 1}, số 0 đã gặp = {cnt}")

x=1, x^1=0, số 0 đã gặp = 0
x=0, x^1=1, số 0 đã gặp = 1
x=1, x^1=0, số 0 đã gặp = 1
x=1, x^1=0, số 0 đã gặp = 1
x=0, x^1=1, số 0 đã gặp = 2


### 3.4. Chỉ số âm trong Python **vòng** về cuối

`a[-1]` là phần tử cuối. Hệ quả nguy hiểm: nếu ta vô tình truy cập `a[i - 1]` với `i = 0`, ta lấy `a[-1]` thay vì báo lỗi — **sai ngầm**. Khi viết mảng "bên trái/bên phải", phải xử lý biên tường minh.

In [5]:
a = [10, 20, 30]
i = 0
print(a[i - 1])      # 30  (!!) lấy phần tử cuối, không phải "không có phần tử bên trái"

30


### 3.5. Generator/iterator và luồng dữ liệu

Một **iterator** cho ta từng giá trị một khi gọi `next(it)` hoặc đi qua bằng vòng `for`, **không lưu cả dãy**. Generator (hàm có `yield`) tạo iterator rất gọn. Nhờ đó ta mô phỏng "luồng vô hạn": chương trình chỉ thấy từng số một và không thể quay lại.

In [6]:
def luong_ngau_nhien(seed, so_luong):
    import random
    rng = random.Random(seed)
    for _ in range(so_luong):
        yield rng.randint(0, 1)          # phát từng số 0/1, không lưu mảng

it = luong_ngau_nhien(7, 8)
print(next(it), next(it), next(it))
print(list(it))                           # phần còn lại

1 0 1
[0, 0, 0, 1, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums = [1, 0, 1, 1, 0]`. Có hai số `0` (chỉ số `1` và `4`). Ta thử từng cách:

- Lật chỉ số `1`: `[1, 1, 1, 1, 0]` → đoạn `1` dài nhất `4`.
- Lật chỉ số `4`: `[1, 0, 1, 1, 1]` → dài nhất `3`.
- Kết quả `max(4, 3) = 4`.

**Quan sát:** lật một số `0` ở chỉ số `z` nối **chuỗi `1` bên trái** và **chuỗi `1` bên phải** của nó với nhau:

`độ dài = (số 1 liên tiếp ngay trước z) + 1 + (số 1 liên tiếp ngay sau z)`.

Với `z = 1`: trái = `1`, phải = `2` ⇒ `1 + 1 + 2 = 4`. Với `z = 4`: trái = `2`, phải = `0` ⇒ `2 + 1 + 0 = 3`. ✓.

Nếu mảng **không có số `0`** nào thì đáp án là cả mảng `n` (không cần lật).

Cách nhìn thứ hai (cửa sổ): đoạn dài nhất chứa **không quá một số `0`**. Với `[1,0,1,1,0]` đó là `nums[0..3] = [1,0,1,1]` (một số `0`) dài `4`.

In [7]:
nums = [1, 0, 1, 1, 0]
for z in [i for i, x in enumerate(nums) if x == 0]:
    trai = 0
    i = z - 1
    while i >= 0 and nums[i] == 1:
        trai += 1
        i -= 1
    phai = 0
    i = z + 1
    while i < len(nums) and nums[i] == 1:
        phai += 1
        i += 1
    print(f"lật chỉ số {z}: trái={trai}, phải={phai}, tổng = {trai + 1 + phai}")

lật chỉ số 1: trái=1, phải=2, tổng = 4
lật chỉ số 4: trái=2, phải=0, tổng = 3


## 5. Cách 1 — Brute Force / baseline: thử lật từng số `0`, đếm lại chuỗi `1` dài nhất

**Ý tưởng:** bám sát đề. Với **mỗi** số `0` (và cả trường hợp không lật nếu không có số `0`), tạo bản sao mảng, lật số `0` đó, rồi quét tìm chuỗi `1` liên tiếp dài nhất. Lấy lớn nhất.

Đây là **oracle** độc lập cho các cách sau. Ta cũng viết thêm một oracle thứ hai bám sát cách nhìn "đoạn dài nhất có tối đa một số `0`" bằng cách xét **mọi** đoạn `[l, r]`.

In [8]:
def chuoi_1_dai_nhat(a):
    best = run = 0
    for x in a:
        run = run + 1 if x == 1 else 0
        best = max(best, run)
    return best

def vet_can(nums):
    ung_vien = [i for i, x in enumerate(nums) if x == 0] or [None]     # None: không có số 0 để lật
    best = 0
    for z in ung_vien:
        b = nums[:]
        if z is not None:
            b[z] = 1
        best = max(best, chuoi_1_dai_nhat(b))
    return best

def moi_doan(nums):
    """Oracle 2: mọi đoạn [l, r] có tối đa một số 0; lấy độ dài lớn nhất."""
    n = len(nums)
    best = 0
    for l in range(n):
        for r in range(l, n):
            if nums[l:r + 1].count(0) <= 1:
                best = max(best, r - l + 1)
    return best

for nums, kq in vi_du:
    print(nums, "->", vet_can(nums), moi_doan(nums))
    assert vet_can(nums) == moi_doan(nums) == kq

[1, 0, 1, 1, 0] -> 4 4
[1, 0, 1, 1, 0, 1] -> 4 4


**Độ phức tạp:** với `z` số `0`, ta quét `O(n)` cho mỗi số `0` ⇒ `O(n · z)` ≤ `O(n²)` (xấu nhất toàn `0`), cộng việc sao chép mảng. Bộ nhớ `O(n)`.

**Nút thắt:** với `n = 10^5` và mảng toàn `0`, cỡ `10^10` thao tác. Hãy đo để thấy tốc độ tăng theo `n`:

In [9]:
import time
for n in (200, 400, 800, 1600):
    a = [0] * n
    t0 = time.perf_counter()
    vet_can(a)
    print(f"n = {n:>5} (toàn 0): {time.perf_counter() - t0:.4f}s")

n =   200 (toàn 0): 0.0024s
n =   400 (toàn 0): 0.0073s
n =   800 (toàn 0): 0.0292s
n =  1600 (toàn 0): 0.1084s


Thời gian tăng gấp ~4 khi `n` tăng gấp đôi — đúng quy luật `O(n²)`. Mỗi số `0` bị xử lý **độc lập** nên ta quét lại các số `1` quanh nó nhiều lần. Ta cần cách gộp thông tin.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Chuỗi `1` liền kề số `0` có thể tính sẵn trong một lượt:** `left[i]` = số `1` liên tiếp **kết thúc** tại `i`, `right[i]` = số `1` liên tiếp **bắt đầu** tại `i`. Khi đó lật số `0` tại `z` cho `left[z-1] + 1 + right[z+1]` trong `O(1)`.
2. **Cửa sổ trượt:** đoạn dài nhất có **tối đa một số `0`**. Mở rộng đầu phải `r` từng bước, đếm số `0`; khi vượt quá một số `0` thì **thu hẹp** đầu trái `l` cho tới khi hợp lệ lại.
3. **Mỗi chỉ số `l`, `r` chỉ tiến, không lùi ⇒ `O(n)`.**
4. **Mẹo "cửa sổ không co":** vì ta chỉ cần **độ dài lớn nhất**, ta có thể để cửa sổ **chỉ lớn lên hoặc trượt**: khi vi phạm, đẩy `l` thêm **một** bước (thay vì cả vòng lặp); độ dài cửa sổ không bao giờ nhỏ đi, và độ dài cuối cùng chính là đáp án.
5. **Với luồng dữ liệu:** ta chỉ cần nhớ **hai số**: chuỗi `1` hiện tại (`cur`) và độ dài đoạn hợp lệ tính **đến số `0` gần nhất đã lật** (`prev`). Bộ nhớ `O(1)`, không cần lưu mảng.

## 7. Các cách tiếp theo

### Cách 2 — Mảng độ dài chuỗi `1` bên trái / bên phải

**Giải thích từng đoạn:**

- `left[i]`: nếu `nums[i] == 1` thì `left[i] = left[i-1] + 1` (với `i = 0` thì `1`), ngược lại `0`.
- `right[i]`: nếu `nums[i] == 1` thì `right[i] = right[i+1] + 1` (với `i = n-1` thì `1`), ngược lại `0`.
- Với **mỗi** vị trí `i` có `nums[i] == 0`: `trai = left[i-1] if i > 0 else 0`; `phai = right[i+1] if i < n-1 else 0`; ứng viên `trai + 1 + phai`.
- Nếu không có số `0` nào thì đáp án là `n`; ta bao quát trường hợp này bằng cách khởi tạo `best = max(left)` (khi toàn `1`, `max(left) = n`).

In [10]:
def trai_phai(nums):
    n = len(nums)
    left = [0] * n
    right = [0] * n
    for i in range(n):
        if nums[i] == 1:
            left[i] = (left[i - 1] if i > 0 else 0) + 1
    for i in range(n - 1, -1, -1):
        if nums[i] == 1:
            right[i] = (right[i + 1] if i < n - 1 else 0) + 1
    best = max(left)                           # trường hợp không có số 0 (hoặc đoạn toàn 1)
    for i in range(n):
        if nums[i] == 0:
            trai = left[i - 1] if i > 0 else 0
            phai = right[i + 1] if i < n - 1 else 0
            best = max(best, trai + 1 + phai)
    return best

for nums, kq in vi_du:
    print(nums, "->", trai_phai(nums))
    assert trai_phai(nums) == kq
# xem hai mảng trung gian của [1, 0, 1, 1, 0]
a = [1, 0, 1, 1, 0]
n = len(a)
left, right = [0] * n, [0] * n
for i in range(n):
    if a[i] == 1:
        left[i] = (left[i - 1] if i > 0 else 0) + 1
for i in range(n - 1, -1, -1):
    if a[i] == 1:
        right[i] = (right[i + 1] if i < n - 1 else 0) + 1
print("nums :", a)
print("left :", left)
print("right:", right)

[1, 0, 1, 1, 0] -> 4
[1, 0, 1, 1, 0, 1] -> 4
nums : [1, 0, 1, 1, 0]
left : [1, 0, 1, 2, 0]
right: [1, 0, 2, 1, 0]


**Code nộp LeetCode (Cách 2):**

```python
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        n = len(nums)
        left = [0] * n
        right = [0] * n
        for i in range(n):
            if nums[i] == 1:
                left[i] = (left[i - 1] if i > 0 else 0) + 1
        for i in range(n - 1, -1, -1):
            if nums[i] == 1:
                right[i] = (right[i + 1] if i < n - 1 else 0) + 1
        best = max(left)
        for i in range(n):
            if nums[i] == 0:
                trai = left[i - 1] if i > 0 else 0
                phai = right[i + 1] if i < n - 1 else 0
                best = max(best, trai + 1 + phai)
        return best
```

**Độ phức tạp:** ba lượt quét ⇒ thời gian `O(n)`, bộ nhớ `O(n)` cho hai mảng.

### Cách 3 — Cửa sổ trượt: co cửa sổ bằng vòng `while` khi có hơn một số `0`

**Giải thích từng đoạn:**

- `l = 0` là đầu trái cửa sổ, `zeros` là số `0` trong cửa sổ `[l..r]`, `best = 0`.
- Duyệt `r = 0 .. n-1`: nếu `nums[r] == 0` thì `zeros += 1`.
- `while zeros > 1:` cửa sổ có hai số `0` (vi phạm). Nếu `nums[l] == 0` thì `zeros -= 1`; rồi `l += 1`. Lặp tới khi `zeros <= 1`.
- Cập nhật `best = max(best, r - l + 1)`.

In [11]:
def cua_so_co(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > 1:
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

for nums, kq in vi_du:
    print(nums, "->", cua_so_co(nums))
    assert cua_so_co(nums) == kq

[1, 0, 1, 1, 0] -> 4
[1, 0, 1, 1, 0, 1] -> 4


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        l = zeros = best = 0
        for r, x in enumerate(nums):
            if x == 0:
                zeros += 1
            while zeros > 1:
                if nums[l] == 0:
                    zeros -= 1
                l += 1
            best = max(best, r - l + 1)
        return best
```

**Độ phức tạp:** `l` và `r` chỉ đi tới, tổng `≤ 2n` bước ⇒ thời gian `O(n)`; bộ nhớ `O(1)`.

### Cách 4 — Cửa sổ "không co": đẩy `l` đúng một bước khi vi phạm

**Ý tưởng khác:** ta chỉ cần **độ dài lớn nhất**, nên không cần co cửa sổ tới khi hợp lệ. Khi số `0` thứ hai xuất hiện, ta **trượt** cả cửa sổ sang phải một bước (`l += 1`, trừ `nums[l]` nếu nó là `0`). Cửa sổ có thể tạm thời "không hợp lệ" nhưng **độ dài nó giữ không đổi** và chỉ tăng khi ta tìm được đoạn hợp lệ dài hơn. Cuối vòng lặp, độ dài cửa sổ `n - l` chính là đáp án.

**Giải thích từng đoạn:**

- `cnt += x ^ 1` (đếm số `0`).
- `if cnt > 1:` thì `cnt -= nums[l] ^ 1` và `l += 1` (đúng **một** lần, không dùng `while`).
- `return n - l`.

In [12]:
def cua_so_khong_co(nums):
    l = cnt = 0
    for x in nums:
        cnt += x ^ 1
        if cnt > 1:
            cnt -= nums[l] ^ 1
            l += 1
    return len(nums) - l

for nums, kq in vi_du:
    print(nums, "->", cua_so_khong_co(nums))
    assert cua_so_khong_co(nums) == kq

[1, 0, 1, 1, 0] -> 4
[1, 0, 1, 1, 0, 1] -> 4


**Code nộp LeetCode (Cách 4):**

```python
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        l = cnt = 0
        for x in nums:
            cnt += x ^ 1
            if cnt > 1:
                cnt -= nums[l] ^ 1
                l += 1
        return len(nums) - l
```

**Độ phức tạp:** một lượt quét ⇒ thời gian `O(n)`, bộ nhớ `O(1)`.

**Vì sao đúng (tóm tắt, chi tiết ở mục 10):** độ dài cửa sổ `r - l + 1` không bao giờ giảm; nó tăng thêm 1 mỗi khi ta mở rộng mà vẫn hợp lệ. Khi vi phạm, cả cửa sổ chỉ trượt (giữ nguyên độ dài). Ta nắm được "độ dài lớn nhất đã đạt" nhờ nó không giảm.

### Cách 5 — Bộ đếm hai trạng thái cho luồng dữ liệu (trả lời follow-up)

**Ý tưởng khác:** không dùng mảng, không dùng chỉ số `l`. Chỉ lưu:

- `cur`: số `1` liên tiếp ngay **sau** số `0` gần nhất (hoặc từ đầu nếu chưa có `0`);
- `prev`: độ dài đoạn hợp lệ **tính tới và gồm số `0` gần nhất** (số `0` đó được lật), tức `(số 1 trước nó) + 1`.

Mỗi số đến:

- Gặp `1`: `cur += 1`.
- Gặp `0`: số `0` này sẽ là số được lật: `prev = cur + 1` (chuỗi `1` vừa rồi + chính số `0` lật), rồi `cur = 0`.
- Đáp án tạm thời: `best = max(best, prev + cur)`.

(Khi chưa gặp `0`, `prev = 0`, `best = cur`.) Bộ nhớ chỉ ba biến, mỗi số được xử lý một lần rồi bỏ.

In [13]:
def dem_hai_trang_thai(luong):
    prev = cur = best = 0
    for x in luong:                # luong có thể là iterator, không cần mảng
        if x == 1:
            cur += 1
        else:
            prev = cur + 1
            cur = 0
        best = max(best, prev + cur)
    return best

for nums, kq in vi_du:
    print(nums, "->", dem_hai_trang_thai(nums))
    assert dem_hai_trang_thai(nums) == kq
print("trên iterator:", dem_hai_trang_thai(iter([1, 0, 1, 1, 0])))

[1, 0, 1, 1, 0] -> 4
[1, 0, 1, 1, 0, 1] -> 4
trên iterator: 4


**Một lớp xử lý luồng thật sự** (gọi `them(x)` từng số, hỏi `ket_qua()` bất cứ lúc nào):

In [14]:
class LuongMotSoKhong:
    def __init__(self):
        self.prev = 0
        self.cur = 0
        self.best = 0

    def them(self, x):
        if x == 1:
            self.cur += 1
        else:
            self.prev = self.cur + 1
            self.cur = 0
        self.best = max(self.best, self.prev + self.cur)

    def ket_qua(self):
        return self.best

s = LuongMotSoKhong()
for x in [1, 0, 1, 1, 0, 1]:
    s.them(x)
    print(f"nhận {x}: kết quả tạm thời = {s.ket_qua()}")
assert s.ket_qua() == 4

nhận 1: kết quả tạm thời = 1
nhận 0: kết quả tạm thời = 2
nhận 1: kết quả tạm thời = 3
nhận 1: kết quả tạm thời = 4
nhận 0: kết quả tạm thời = 4
nhận 1: kết quả tạm thời = 4


**Code nộp LeetCode (Cách 5):**

```python
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        prev = cur = best = 0
        for x in nums:
            if x == 1:
                cur += 1
            else:
                prev = cur + 1
                cur = 0
            best = max(best, prev + cur)
        return best
```

**Độ phức tạp:** thời gian `O(n)`, bộ nhớ `O(1)`, và **đúng cho luồng vô hạn** vì không cần lưu quá khứ.

## 8. TRACE từng bước

Dùng Cách 3 (cửa sổ trượt) với `nums = [1, 0, 1, 1, 0, 1, 1, 1]` (ví dụ tự dựng, đối chiếu lại với oracle ở mục 12). Đáp án là `6`: lật số `0` ở chỉ số `4` nối hai chuỗi `1, 1` (chỉ số 2, 3) và `1, 1, 1` (chỉ số 5, 6, 7), được `2 + 1 + 3 = 6`.

| r | nums[r] | zeros | l sau khi co | cửa sổ | độ dài | best |
|---|---|---|---|---|---|---|
| 0 | 1 | 0 | 0 | [1] | 1 | 1 |
| 1 | 0 | 1 | 0 | [1,0] | 2 | 2 |
| 2 | 1 | 1 | 0 | [1,0,1] | 3 | 3 |
| 3 | 1 | 1 | 0 | [1,0,1,1] | 4 | 4 |
| 4 | 0 | 2 → co | 2 (bỏ `1`, bỏ `0`) | [1,1,0] | 3 | 4 |
| 5 | 1 | 1 | 2 | [1,1,0,1] | 4 | 4 |
| 6 | 1 | 1 | 2 | [1,1,0,1,1] | 5 | 5 |
| 7 | 1 | 1 | 2 | [1,1,0,1,1,1] | 6 | 6 |

In [15]:
def trace_cua_so(nums):
    l = zeros = best = 0
    print("nums =", nums)
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        co = 0
        while zeros > 1:
            if nums[l] == 0:
                zeros -= 1
            l += 1
            co += 1
        best = max(best, r - l + 1)
        print(f"r={r}, nums[r]={x}, co {co} bước -> l={l}, zeros={zeros}, cửa sổ={nums[l:r+1]}, dài={r - l + 1}, best={best}")
    return best

assert trace_cua_so([1, 0, 1, 1, 0, 1, 1, 1]) == 6

nums = [1, 0, 1, 1, 0, 1, 1, 1]
r=0, nums[r]=1, co 0 bước -> l=0, zeros=0, cửa sổ=[1], dài=1, best=1
r=1, nums[r]=0, co 0 bước -> l=0, zeros=1, cửa sổ=[1, 0], dài=2, best=2
r=2, nums[r]=1, co 0 bước -> l=0, zeros=1, cửa sổ=[1, 0, 1], dài=3, best=3
r=3, nums[r]=1, co 0 bước -> l=0, zeros=1, cửa sổ=[1, 0, 1, 1], dài=4, best=4
r=4, nums[r]=0, co 2 bước -> l=2, zeros=1, cửa sổ=[1, 1, 0], dài=3, best=4
r=5, nums[r]=1, co 0 bước -> l=2, zeros=1, cửa sổ=[1, 1, 0, 1], dài=4, best=4
r=6, nums[r]=1, co 0 bước -> l=2, zeros=1, cửa sổ=[1, 1, 0, 1, 1], dài=5, best=5
r=7, nums[r]=1, co 0 bước -> l=2, zeros=1, cửa sổ=[1, 1, 0, 1, 1, 1], dài=6, best=6


**Trace Cách 4 (không co)** trên cùng mảng. Để ý `l` chỉ đứng yên hoặc nhích một bước, và `n - l` là đáp án cuối cùng:

In [16]:
def trace_khong_co(nums):
    l = cnt = 0
    for r, x in enumerate(nums):
        cnt += x ^ 1
        if cnt > 1:
            cnt -= nums[l] ^ 1
            l += 1
        print(f"r={r}, x={x}, cnt={cnt}, l={l}, độ dài cửa sổ = r - l + 1 = {r - l + 1}")
    print("đáp án = n - l =", len(nums) - l)
    return len(nums) - l

assert trace_khong_co([1, 0, 1, 1, 0, 1, 1, 1]) == 6

r=0, x=1, cnt=0, l=0, độ dài cửa sổ = r - l + 1 = 1
r=1, x=0, cnt=1, l=0, độ dài cửa sổ = r - l + 1 = 2
r=2, x=1, cnt=1, l=0, độ dài cửa sổ = r - l + 1 = 3
r=3, x=1, cnt=1, l=0, độ dài cửa sổ = r - l + 1 = 4
r=4, x=0, cnt=2, l=1, độ dài cửa sổ = r - l + 1 = 4
r=5, x=1, cnt=1, l=2, độ dài cửa sổ = r - l + 1 = 4
r=6, x=1, cnt=1, l=2, độ dài cửa sổ = r - l + 1 = 5
r=7, x=1, cnt=1, l=2, độ dài cửa sổ = r - l + 1 = 6
đáp án = n - l = 6


**Trace Cách 5 (hai trạng thái)** trên cùng mảng:

In [17]:
def trace_hai_trang_thai(nums):
    prev = cur = best = 0
    for x in nums:
        if x == 1:
            cur += 1
        else:
            prev = cur + 1
            cur = 0
        best = max(best, prev + cur)
        print(f"x={x}: prev={prev}, cur={cur}, prev+cur={prev + cur}, best={best}")
    return best

assert trace_hai_trang_thai([1, 0, 1, 1, 0, 1, 1, 1]) == 6

x=1: prev=0, cur=1, prev+cur=1, best=1
x=0: prev=2, cur=0, prev+cur=2, best=2
x=1: prev=2, cur=1, prev+cur=3, best=3
x=1: prev=2, cur=2, prev+cur=4, best=4
x=0: prev=3, cur=0, prev+cur=3, best=4
x=1: prev=3, cur=1, prev+cur=4, best=4
x=1: prev=3, cur=2, prev+cur=5, best=5
x=1: prev=3, cur=3, prev+cur=6, best=6


## 9. Code nộp LeetCode

Bản dùng Cách 4 (cửa sổ không co), đúng `class Solution` và chữ ký:

In [18]:
from typing import List

class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        l = cnt = 0
        for x in nums:
            cnt += x ^ 1
            if cnt > 1:
                cnt -= nums[l] ^ 1
                l += 1
        return len(nums) - l

sol = Solution()
print(sol.findMaxConsecutiveOnes([1, 0, 1, 1, 0]), sol.findMaxConsecutiveOnes([1, 0, 1, 1, 0, 1]))
assert sol.findMaxConsecutiveOnes([1, 0, 1, 1, 0]) == 4
assert sol.findMaxConsecutiveOnes([1, 0, 1, 1, 0, 1]) == 4

4 4


## 10. Vì sao thuật toán đúng

**Bài toán tương đương.** Lật tối đa một số `0` ⇒ đoạn toàn `1` dài nhất = **đoạn dài nhất chứa không quá một số `0`**. (Đoạn như vậy, sau khi lật số `0` duy nhất nếu có, thành toàn `1`; ngược lại mọi đoạn toàn `1` sau khi lật tối đa một số `0` chứa tối đa một số `0` ở trạng thái gốc.)

**Cách 3 (co cửa sổ).** Bất biến: sau mỗi bước `r`, `[l..r]` là đoạn dài nhất kết thúc tại `r` chứa không quá một số `0`. Vì khi `zeros > 1` ta chỉ tăng `l` đủ để loại số `0` đầu tiên (đủ đưa `zeros` về 1), mà đoạn kết thúc tại `r` càng ngắn khi `l` càng lớn nên ta chọn `l` nhỏ nhất hợp lệ. Lấy `max` qua mọi `r` ⇒ đáp án.

**Cách 4 (không co).** Gọi `W_r = r - l + 1` là độ dài cửa sổ sau bước `r`. Bước đầu `W` tăng 1 mỗi lần cho tới khi gặp số `0` thứ hai. Từ đó, mỗi bước: `r` tăng 1 và `l` tăng 0 hoặc 1. Nếu cửa sổ mới **không** vi phạm (`cnt ≤ 1`) thì `l` không đổi và `W` tăng 1 — tức ta đã tìm được đoạn **hợp lệ** dài hơn, đúng chỗ để tăng. Nếu vi phạm, `l` tăng 1 và `W` giữ nguyên. Hai tính chất:

- `W` không bao giờ giảm, và nó chỉ tăng khi cửa sổ mới hợp lệ dài `W+1`. Do đó `W` luôn bằng độ dài hợp lệ lớn nhất đã thấy.
- Cửa sổ mà ta giữ nguyên độ dài có thể **chứa hai số `0`**, nhưng độ dài của nó không vượt độ dài hợp lệ lớn nhất đã tìm được nên không làm sai đáp án.

Vì vậy giá trị cuối `n - l` bằng độ dài hợp lệ lớn nhất.

**Cách 5 (hai trạng thái).** Mỗi đoạn hợp lệ kết thúc tại vị trí hiện tại có một trong hai dạng: (a) không có số `0` nào — dài `cur` (chỉ khi chưa từng gặp `0`); (b) có đúng một số `0` — là số `0` gần nhất, khi đó đoạn dài nhất là `(chuỗi 1 trước số 0 đó) + 1 + (chuỗi 1 sau nó)`, tức `prev + cur`. Chuỗi `1` trước số `0` được chốt khi gặp `0` (`prev = cur + 1`) và `cur` làm mới. Số `0` cũ hơn không thể lật cùng lúc nên bị bỏ.

**Cách 2** đúng vì mỗi đoạn tối ưu có dạng "trái + 1 + phải" quanh một số `0` duy nhất, hoặc là đoạn toàn `1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử lật từng số 0 | `O(n · z)` ≤ `O(n²)` | `O(n)` |
| 2. Mảng trái/phải | `O(n)` | `O(n)` |
| 3. Cửa sổ co (`while`) | `O(n)` | `O(1)` |
| 4. Cửa sổ không co | `O(n)` | `O(1)` |
| 5. Hai trạng thái (stream) | `O(n)` | `O(1)` |

Vì sao Cách 3 là `O(n)` dù có `while` lồng trong `for`: `l` chỉ tăng, tối đa `n` lần trong **cả** quá trình (không phải `n` lần mỗi vòng `for`). Tổng số bước `r` cộng `l` ≤ `2n`.

In [19]:
import random
random.seed(487)
n = 100000
a = [random.randint(0, 1) for _ in range(n)]
for ten, f in [("trai_phai", trai_phai), ("cua_so_co", cua_so_co),
               ("cua_so_khong_co", cua_so_khong_co), ("hai_trang_thai", dem_hai_trang_thai)]:
    t0 = time.perf_counter()
    kq = f(a)
    print(f"{ten:16}: {kq}, {time.perf_counter() - t0:.4f}s")

trai_phai       : 19, 0.0257s
cua_so_co       : 19, 0.0221s
cua_so_khong_co : 19, 0.0113s
hai_trang_thai  : 19, 0.0058s


## 12. Bộ test

Chiến lược: ví dụ đề; biên (toàn `1`, toàn `0`, một phần tử, `0` ở đầu/cuối); phản ví dụ của vài "mẹo" sai; **đối chiếu hai oracle vét cạn** với mọi mảng nhị phân độ dài ≤ 12; ngẫu nhiên có seed đúng ràng buộc; kiểm tra **luồng** bằng iterator.

In [20]:
import itertools

CACH = {
    "trai_phai": trai_phai,
    "cua_so_co": cua_so_co,
    "cua_so_khong_co": cua_so_khong_co,
    "hai_trang_thai": dem_hai_trang_thai,
    "nop": Solution().findMaxConsecutiveOnes,
}

# 1. Ví dụ đề + biên
tests = [([1, 0, 1, 1, 0], 4), ([1, 0, 1, 1, 0, 1], 4),
         ([1], 1), ([0], 1), ([1, 1, 1], 3), ([0, 0], 1), ([0, 1], 2), ([1, 0], 2),
         ([0, 0, 0, 0], 1), ([1, 1, 0, 1, 1], 5), ([0, 1, 1, 0, 1, 1, 0], 5)]
for nums, kq in tests:
    assert vet_can(nums) == moi_doan(nums) == kq, (nums, vet_can(nums), moi_doan(nums), kq)
    for ten, f in CACH.items():
        assert f(nums) == kq, (ten, nums, f(nums), kq)
print("ví dụ + biên: OK")

ví dụ + biên: OK


In [21]:
# 2. Hai oracle + tất cả cách trên MỌI mảng nhị phân độ dài 1..12 (8190 mảng)
dem = 0
for L in range(1, 13):
    for tup in itertools.product((0, 1), repeat=L):
        nums = list(tup)
        gt = vet_can(nums)
        if L <= 9:
            assert moi_doan(nums) == gt
        for ten, f in CACH.items():
            assert f(nums) == gt, (ten, nums)
        dem += 1
print("khớp oracle trên", dem, "mảng: OK")

khớp oracle trên 8190 mảng: OK


In [22]:
# 3. Ngẫu nhiên có seed, đúng ràng buộc (độ dài <= 10^5), nhiều mật độ số 0
random.seed(20487)
for _ in range(40):
    n = random.randint(1, 20000)
    p0 = random.choice([0.01, 0.1, 0.5, 0.9, 0.99])
    nums = [0 if random.random() < p0 else 1 for _ in range(n)]
    ref = cua_so_co(nums)
    for ten, f in CACH.items():
        assert f(nums) == ref, (ten, n, p0)
print("ngẫu nhiên: OK")

# 4. Luồng: iterator không lưu mảng, so với bản dùng mảng
for seed in range(5):
    it1 = luong_ngau_nhien(seed, 200000)
    kq_luong = dem_hai_trang_thai(it1)
    mang = list(luong_ngau_nhien(seed, 200000))
    assert kq_luong == cua_so_co(mang) == trai_phai(mang)
print("luồng 200000 phần tử: OK")

ngẫu nhiên: OK


luồng 200000 phần tử: OK


**Phản ví dụ cho các "mẹo" sai hay gặp:**

- "Đáp án = chuỗi `1` dài nhất `+ 1`": sai khi **không có số `0`** (toàn `1`: không thể vượt `n`) hoặc khi hai chuỗi cách nhau bởi nhiều số `0`.
- "Đáp án = số số `1` + 1": sai vì cần **liên tiếp**.

In [23]:
def meo_sai_1(nums):
    return chuoi_1_dai_nhat(nums) + 1

def meo_sai_2(nums):
    return min(len(nums), sum(nums) + 1)

for nums in ([1, 1, 1], [1, 0, 0, 1], [1, 1, 0, 0, 0, 1, 1]):
    print(nums, "| đúng:", cua_so_co(nums), "| mẹo 1:", meo_sai_1(nums), "| mẹo 2:", meo_sai_2(nums))
assert meo_sai_1([1, 1, 1]) != cua_so_co([1, 1, 1])
assert meo_sai_2([1, 0, 0, 1]) != cua_so_co([1, 0, 0, 1])

[1, 1, 1] | đúng: 3 | mẹo 1: 4 | mẹo 2: 3
[1, 0, 0, 1] | đúng: 2 | mẹo 1: 2 | mẹo 2: 3
[1, 1, 0, 0, 0, 1, 1] | đúng: 3 | mẹo 1: 3 | mẹo 2: 5


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `chuỗi 1 dài nhất + 1`

**Code sai:** `return chuoi_1_dai_nhat(nums) + 1`.

**Vì sao sai:** nếu mảng toàn `1`, ta **không có số `0` để lật**, nên kết quả không thể vượt `n`; ngoài ra nếu hai chuỗi `1` bị ngăn bởi hai số `0` trở lên thì lật một số `0` không nối được chúng.

In [24]:
print("[1,1,1]:", meo_sai_1([1, 1, 1]), "đúng:", cua_so_co([1, 1, 1]))
print("[1,0,0,1]:", meo_sai_1([1, 0, 0, 1]), "đúng:", cua_so_co([1, 0, 0, 1]))
assert meo_sai_1([1, 1, 1]) == 4 and cua_so_co([1, 1, 1]) == 3

[1,1,1]: 4 đúng: 3
[1,0,0,1]: 2 đúng: 2


**Code đúng:** cửa sổ trượt (Cách 3/4) hoặc trái-phải (Cách 2).

### Lỗi 2 — Điều kiện co cửa sổ sai (`>= 1` thay vì `> 1`)

**Code sai:** `while zeros >= 1:` ⇒ cửa sổ **không chứa số `0` nào**, tức quay về bài "Max Consecutive Ones" thường (không lật).

**Vì sao sai:** ta được phép có **một** số `0` (bị lật) trong cửa sổ.

In [25]:
def sai_dieu_kien(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros >= 1:                        # SAI: phải là > 1
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

print(sai_dieu_kien([1, 0, 1, 1, 0]), "đúng:", cua_so_co([1, 0, 1, 1, 0]))
assert sai_dieu_kien([1, 0, 1, 1, 0]) == 2

2 đúng: 4


### Lỗi 3 — Chỉ số âm vòng về cuối (`left[i - 1]` khi `i = 0`)

**Code sai (Cách 2):** viết `trai = left[i - 1]` mà không kiểm tra `i > 0`. Ở `i = 0`, Python lấy `left[-1]`, tức phần tử **cuối** mảng — **không báo lỗi** nhưng sai kết quả.

In [26]:
def sai_chi_so_am(nums):
    n = len(nums)
    left = [0] * n
    right = [0] * n
    for i in range(n):
        if nums[i] == 1:
            left[i] = (left[i - 1] if i > 0 else 0) + 1
    for i in range(n - 1, -1, -1):
        if nums[i] == 1:
            right[i] = (right[i + 1] if i < n - 1 else 0) + 1
    best = max(left)
    for i in range(n):
        if nums[i] == 0:
            trai = left[i - 1]                        # SAI: i = 0 thì lấy left[-1]
            phai = right[i + 1] if i < n - 1 else 0
            best = max(best, trai + 1 + phai)
    return best

# mảng bắt đầu bằng 0 và kết thúc bằng chuỗi 1 dài: left[-1] bị "đọc nhầm"
a = [0, 1, 0, 0, 1, 1, 1]
print("sai:", sai_chi_so_am(a), "| đúng:", cua_so_co(a))
assert sai_chi_so_am(a) != cua_so_co(a)

sai: 5 | đúng: 4


**Code đúng:** `trai = left[i - 1] if i > 0 else 0`.

### Lỗi 4 — Quên cộng chính số `0` bị lật (lệch một)

**Code sai (Cách 5):** `prev = cur` thay vì `prev = cur + 1`. Ta mất đi **chính ô được lật**.

In [27]:
def sai_quen_cong_1(luong):
    prev = cur = best = 0
    for x in luong:
        if x == 1:
            cur += 1
        else:
            prev = cur                       # SAI: phải là cur + 1
            cur = 0
        best = max(best, prev + cur)
    return best

print(sai_quen_cong_1([1, 0, 1, 1, 0]), "đúng:", dem_hai_trang_thai([1, 0, 1, 1, 0]))
assert sai_quen_cong_1([1, 0, 1, 1, 0]) != dem_hai_trang_thai([1, 0, 1, 1, 0])

3 đúng: 4


**Mẹo tránh lỗi:** thử luôn các mảng `[0]`, `[1]`, `[0, 0]`, `[1, 1, 1]`, `[0, 1]`: chúng lộ ra phần lớn lỗi biên.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đoạn liên tiếp dài nhất sao cho có **tối đa `k`** phần tử xấu" ⇒ **cửa sổ trượt**: mở rộng phải, co trái khi vi phạm.
- **Dạng chuẩn `k = 1`:** chỉ cần một biến đếm (số `0`), có thể thay bằng "vị trí số `0` gần nhất".
- **Mẹo không co:** khi chỉ hỏi **độ dài lớn nhất** (không cần đoạn cụ thể), dùng `if` thay `while` và trả `n - l`.
- **Câu hỏi luồng (stream/follow-up):** hãy tìm các **biến trạng thái nhỏ** đủ để cập nhật từng phần tử (ở đây: `prev`, `cur`).
- **Mảng trái/phải (prefix/suffix)** là kỹ thuật chung khi một phần tử "nối" hai phía; luôn xử lý biên và nhớ chỉ số âm vòng.
- Khi thấy `x ^ 1`, nghĩ "đảo bit" — có thể thay bằng `1 - x` cho dễ đọc.

## 15. Biến thể

1. **Max Consecutive Ones (485):** không được lật ⇒ `k = 0`.
2. **Max Consecutive Ones III (1004):** được lật tối đa `k` số `0` ⇒ cửa sổ với `zeros <= k`.
3. **Longest Subarray of 1's After Deleting One Element (1493):** phải xóa đúng một phần tử ⇒ đáp án = (đoạn dài nhất có tối đa một số 0) `- 1`.
4. **Longest Repeating Character Replacement (424):** cùng khung cửa sổ, đổi điều kiện vi phạm.
5. **Trả về chỉ số số `0` nên lật** để được đoạn dài nhất (thay vì độ dài).

In [28]:
def toi_da_k(nums, k):
    """Biến thể 1004: lật tối đa k số 0."""
    l = zeros = best = 0
    for r, x in enumerate(nums):
        zeros += 1 - x
        while zeros > k:
            zeros -= 1 - nums[l]
            l += 1
        best = max(best, r - l + 1)
    return best

def xoa_mot_phan_tu(nums):
    """Biến thể 1493: phải xóa đúng một phần tử."""
    return cua_so_co(nums) - 1

print(toi_da_k([1, 1, 0, 0, 1, 1, 1, 0, 1], 2), xoa_mot_phan_tu([1, 1, 0, 1]), xoa_mot_phan_tu([1, 1, 1]))
assert toi_da_k([1, 0, 1, 1, 0], 1) == 4 and toi_da_k([1, 0, 1, 1, 0], 0) == 2
assert xoa_mot_phan_tu([1, 1, 0, 1]) == 3 and xoa_mot_phan_tu([1, 1, 1]) == 2

7 3 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử lật từng `0` | vét cạn, quét lại mỗi lần | `O(n·z)` | `O(n)` | oracle kiểm thử |
| 2. Trái/phải | tiền xử lý hai mảng, ghép quanh mỗi `0` | `O(n)` | `O(n)` | dễ hiểu, trực quan |
| 3. Cửa sổ co | `while zeros > 1` | `O(n)` | `O(1)` | khuôn mẫu cửa sổ trượt chuẩn, tổng quát `k` |
| 4. Cửa sổ không co | `if` + `n - l` | `O(n)` | `O(1)` | ngắn nhất, chỉ cần độ dài |
| 5. Hai trạng thái | `prev`, `cur` | `O(n)` | `O(1)` | luồng dữ liệu (follow-up) |

**Khuyên dùng khi thi:** Cách 3 (dễ đúng nhất, mở rộng cho `k` bất kỳ) hoặc Cách 4 (ngắn gọn).

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `cua_so_co([0, 1, 1, 0])`, `trai_phai([1, 1, 1])` và `dem_hai_trang_thai([0, 0, 0])`; sau đó chạy để kiểm tra.

In [29]:
du_doan_1 = None
du_doan_2 = None
du_doan_3 = None
print("Thực tế:", cua_so_co([0, 1, 1, 0]), trai_phai([1, 1, 1]), dem_hai_trang_thai([0, 0, 0]))

Thực tế: 3 3 1


**Bài 2 (tính tay).** Với `nums = [1, 1, 0, 1, 0, 1, 1, 1, 0, 1]`: (a) liệt kê, với mỗi số `0`, giá trị `trái + 1 + phải`; (b) nêu đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `cua_so_todo(nums)` theo Cách 3.

In [30]:
def cua_so_todo(nums):
    l = zeros = best = 0
    # TODO 1: for r, x in enumerate(nums): nếu x == 0 thì zeros += 1
    # TODO 2: while zeros > 1: nếu nums[l] == 0 thì zeros -= 1; l += 1
    # TODO 3: best = max(best, r - l + 1)
    return None

kq_bai3 = None       # sau khi viết xong: kq_bai3 = cua_so_todo([1, 0, 1, 1, 0])

**Bài 4 (tự viết lại).** Không nhìn lời giải, tự viết một lớp `LuongTuViet` có hai phương thức: `them(x)` nhận từng số `0/1` của luồng và `ket_qua()` trả độ dài tốt nhất hiện tại. Chỉ được dùng ba biến số nguyên.

In [31]:
class LuongTuViet:
    def __init__(self):
        pass        # TODO: khởi tạo prev, cur, best

    def them(self, x):
        pass        # TODO

    def ket_qua(self):
        pass        # TODO

**Bài 5 (biến thể / mini challenge).** Viết `chi_so_nen_lat(nums)` trả về **chỉ số** của số `0` mà lật nó cho đoạn `1` liên tiếp dài nhất (nếu hòa chọn chỉ số nhỏ nhất; nếu không có số `0` trả `-1`). Gợi ý ở mục 18.

In [32]:
def chi_so_nen_lat(nums):
    # TODO: viết tự do
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):**
  - Bài 1: `[0,1,1,0]` — đoạn hợp lệ dài nhất có một số `0` là `[0,1,1]`.
  - Bài 3: ba TODO dịch từ ba gạch đầu dòng của Cách 3.
  - Bài 5: dùng Cách 2 (trái/phải), với mỗi số `0` tính `trái + 1 + phải`.
- **Mức 2 (vừa):**
  - Bài 2: các số `0` ở chỉ số `2, 4, 8`: `2+1+1 = 4`, `1+1+3 = 5`, `3+1+1 = 5`.
  - Bài 4: `them`: nếu `x == 1` thì `cur += 1`, ngược lại `prev = cur + 1; cur = 0`; sau đó `best = max(best, prev + cur)`.
  - Bài 5: duyệt `i` tăng dần và dùng dấu `>` (không phải `>=`) để chọn chỉ số nhỏ nhất khi hòa.
- **Mức 3 (gần đáp án):**
  - Bài 2: đáp án `5`.
  - Bài 4: copy hàm `LuongMotSoKhong` ở Cách 5, đổi tên.
  - Bài 5: `best_len = 0; best_i = -1; for i in vi_tri_0: tinh = trai + 1 + phai; if tinh > best_len: best_len, best_i = tinh, i`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert cua_so_co([0, 1, 1, 0]) == 3
assert trai_phai([1, 1, 1]) == 3
assert dem_hai_trang_thai([0, 0, 0]) == 1
print(cua_so_co([0, 1, 1, 0]), trai_phai([1, 1, 1]), dem_hai_trang_thai([0, 0, 0]))

# Bài 2
a = [1, 1, 0, 1, 0, 1, 1, 1, 0, 1]
ung_vien = []
for z in [i for i, x in enumerate(a) if x == 0]:
    t = 0
    i = z - 1
    while i >= 0 and a[i] == 1:
        t += 1; i -= 1
    p = 0
    i = z + 1
    while i < len(a) and a[i] == 1:
        p += 1; i += 1
    ung_vien.append(t + 1 + p)
print(ung_vien)
assert ung_vien == [4, 5, 5]
assert max(ung_vien) == cua_so_co(a) == 5

3 3 1
[4, 5, 5]


In [34]:
# Bài 3
def cua_so_dap_an(nums):
    l = zeros = best = 0
    for r, x in enumerate(nums):
        if x == 0:
            zeros += 1
        while zeros > 1:
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l + 1)
    return best

assert cua_so_dap_an([1, 0, 1, 1, 0]) == 4
for tup in itertools.product((0, 1), repeat=8):
    assert cua_so_dap_an(list(tup)) == vet_can(list(tup))

# Bài 4
class LuongTuVietDapAn:
    def __init__(self):
        self.prev = 0
        self.cur = 0
        self.best = 0

    def them(self, x):
        if x == 1:
            self.cur += 1
        else:
            self.prev = self.cur + 1
            self.cur = 0
        self.best = max(self.best, self.prev + self.cur)

    def ket_qua(self):
        return self.best

for seed in range(5):
    s = LuongTuVietDapAn()
    mang = []
    for x in luong_ngau_nhien(seed, 3000):
        s.them(x)
        mang.append(x)
    assert s.ket_qua() == cua_so_co(mang)

In [35]:
# Bài 5
def chi_so_nen_lat_dap_an(nums):
    n = len(nums)
    left = [0] * n
    right = [0] * n
    for i in range(n):
        if nums[i] == 1:
            left[i] = (left[i - 1] if i > 0 else 0) + 1
    for i in range(n - 1, -1, -1):
        if nums[i] == 1:
            right[i] = (right[i + 1] if i < n - 1 else 0) + 1
    best_len, best_i = 0, -1
    for i in range(n):
        if nums[i] == 0:
            trai = left[i - 1] if i > 0 else 0
            phai = right[i + 1] if i < n - 1 else 0
            if trai + 1 + phai > best_len:
                best_len, best_i = trai + 1 + phai, i
    return best_i

assert chi_so_nen_lat_dap_an([1, 0, 1, 1, 0]) == 1
assert chi_so_nen_lat_dap_an([1, 1, 1]) == -1
# kiểm chứng bằng cách lật thật
for tup in itertools.product((0, 1), repeat=8):
    a = list(tup)
    i = chi_so_nen_lat_dap_an(a)
    if i == -1:
        assert 0 not in a
    else:
        b = a[:]
        b[i] = 1
        assert chuoi_1_dai_nhat(b) == vet_can(a)
        # chỉ số nhỏ nhất trong các chỉ số tối ưu
        for z in range(i):
            if a[z] == 0:
                c = a[:]
                c[z] = 1
                assert chuoi_1_dai_nhat(c) < vet_can(a)
print("Tất cả đáp án OK")

Tất cả đáp án OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (cửa sổ trượt, tối đa một số `0`):**

```python
class Solution:
    def findMaxConsecutiveOnes(self, nums: List[int]) -> int:
        l = zeros = best = 0
        for r, x in enumerate(nums):
            if x == 0:
                zeros += 1
            while zeros > 1:          # vượt quá một số 0 thì co trái
                if nums[l] == 0:
                    zeros -= 1
                l += 1
            best = max(best, r - l + 1)
        return best
```

**Template ngắn (cửa sổ không co):** `cnt += x ^ 1; if cnt > 1: cnt -= nums[l] ^ 1; l += 1` rồi `return len(nums) - l`.

**Template luồng:** `prev = cur = best = 0; x == 1: cur += 1; x == 0: prev = cur + 1; cur = 0; best = max(best, prev + cur)`.

**Checklist trước khi submit:**

- [ ] Hiểu đề thành "đoạn dài nhất có **tối đa một** số `0`".
- [ ] Điều kiện co cửa sổ là `zeros > 1` (không phải `>= 1`).
- [ ] Không dùng `max_run + 1` (toàn `1` thì sao?).
- [ ] Nếu dùng mảng trái/phải: xử lý `i = 0` và `i = n - 1` tường minh (chú ý chỉ số âm vòng).
- [ ] Ở bộ đếm hai trạng thái: `prev = cur + 1` (cộng cả ô được lật).
- [ ] Đã thử `[0]`, `[1]`, `[0, 0]`, `[1, 1, 1]`, `[0, 1]`, `[1, 0]`.
- [ ] Trả lời follow-up: Cách 5 chạy trên luồng với bộ nhớ `O(1)`.
- [ ] Độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ.